In [ ]:
# ===== 全局运行配置 =====
MODE = "demo"          # "eval" 或 "demo"
DATASET = "cnc"        # "cnc" 或 "li"（eval 模式 + demo sample 模式使用）

# demo 模式专用
DEMO_INPUT = "manual"     # "manual"（手动输入单条）或 "sample"（数据集抽样）
DEMO_TEXT = "Heavy rain caused widespread flooding in the region."  # manual 时使用
DEMO_SAMPLE_N = 10        # sample 时使用：从数据集随机抽 N 条

# RAG 开关与模式
USE_RAG = False
RAG_MODE = "pattern"   # "pattern"、"knn" 或 "knn_pattern"
RAG_TOP_K = 3

# LLM 参数
TEMPERATURE = 0.0      # eval 用 0 保证可复现；demo 可以调高看多样性
MAX_TOKENS = 2048


In [ ]:
# ===== SPEC_02 验收 =====
# Cell A：导入、环境检查与初始化
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Python executable: {sys.executable}")
if "master_thesis" not in sys.executable.lower():
    raise RuntimeError(
        "当前 Notebook kernel 不是 Master_thesis。请在 Kernel -> Change Kernel 中选择 Master_thesis，"
        "然后 Restart Kernel 后从第一个 cell 重新运行。"
    )

try:
    import openai
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "当前 kernel 缺少 openai，说明 notebook 没有运行在 Master_thesis 环境里，"
        "或 kernel 需要重启。请切换到 Master_thesis 并 Restart Kernel。"
    ) from exc

from src.llm_client import LLMClient
client = LLMClient(temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
print(f"Base URL: {client.base_url}")
print(f"Model: {client.model}")


In [ ]:
# Cell B：连通性测试
assert client.ping(), "LM Studio server 未启动或无法访问"
print("✓ LM Studio 连接正常")


In [ ]:
# Cell C：最简单的 hello 测试
resp = client.chat([{"role": "user", "content": "Reply with exactly the word 'hello' and nothing else."}])
print(f"模型回复: {resp}")


In [ ]:
# Cell D：System message 测试
resp = client.chat([
    {"role": "system", "content": "You are a translator. Translate user input to French."},
    {"role": "user", "content": "Good morning"}
])
print(f"翻译结果: {resp}")


In [ ]:
# Cell E：真实因果抽取试探（只是测 client，不评估输出质量）
resp = client.chat([
    {"role": "user", "content": "Extract cause and effect from this sentence: 'Smoking causes lung cancer.' Output in JSON format."}
])
print(f"因果输出: {resp}")


In [ ]:
# ===== SPEC_03 验收 =====
# Cell F：验证 Pattern Database 与 BGE embedding cache 文件存在
from pathlib import Path

pattern_db_path = PROJECT_ROOT / "RAG Database" / "comb_SCITEsemADE_CausalityPattern.csv"
bge_metadata_path = PROJECT_ROOT / "RAG Database" / "bge-small-en-v1.5_examples.jsonl"
bge_embeddings_path = PROJECT_ROOT / "RAG Database" / "bge-small-en-v1.5_embeddings.npy"
assert pattern_db_path.exists(), f"Pattern DB 不存在: {pattern_db_path}"
print(f"OK Pattern Database 已就绪: {pattern_db_path}")
print(f"  文件大小: {pattern_db_path.stat().st_size / 1024:.1f} KB")

if USE_RAG and RAG_MODE in {"knn", "knn_pattern"}:
    assert bge_metadata_path.exists(), f"BGE metadata cache 不存在: {bge_metadata_path}"
    assert bge_embeddings_path.exists(), f"BGE embeddings cache 不存在: {bge_embeddings_path}"
    print(f"OK BGE metadata cache 已就绪: {bge_metadata_path}")
    print(f"OK BGE embeddings cache 已就绪: {bge_embeddings_path}")
print(f"RAG mode: {RAG_MODE} | USE_RAG={USE_RAG} | top_k={RAG_TOP_K}")


In [ ]:
# Cell G：检验 Prompt 模板渲染（不调 LLM）
from src.prompt_builder import build_messages

messages = build_messages(DEMO_TEXT, use_rag=False, retriever=None, top_k=0)
for m in messages:
    print(f"--- {m['role']} ---")
    print(m['content'])
    print()


In [ ]:
# Cell H：按 RAG_MODE 渲染 prompt
from src.retriever_factory import create_retriever

retriever = None
if USE_RAG:
    retriever = create_retriever(
        RAG_MODE,
        metadata_path=bge_metadata_path,
        embeddings_path=bge_embeddings_path,
    )

messages_rag = build_messages(
    DEMO_TEXT,
    use_rag=USE_RAG,
    retriever=retriever,
    top_k=RAG_TOP_K,
    rag_mode=RAG_MODE,
)
for m in messages_rag:
    print(f"--- {m['role']} ---")
    print(m['content'])
    print()


In [ ]:
# Cell I：观察当前 RAG_MODE 检索到的 examples
if retriever is None:
    print("RAG 未启用；如需测试检索，请设置 USE_RAG=True，并选择 RAG_MODE。")
else:
    examples = retriever.retrieve(DEMO_TEXT, top_k=RAG_TOP_K)
    print(f"检索到 {len(examples)} 个 examples:")
    for i, ex in enumerate(examples, 1):
        print(f"\n--- Example {i} ---")
        print(ex)


In [ ]:
# Cell J：用 SPEC_02 的 client 真正调一次 LLM（只看输出，不解析）
resp = client.chat(messages_rag if USE_RAG else messages)
print("=== LLM 原始输出 ===")
print(resp)


In [ ]:
# ===== SPEC_04 验收 =====
# Cell K：parse_output 鲁棒性快速测试（不调 LLM，用构造字符串）
import json
from src.generator import generate, parse_output
from src.data_io import load_dataset

test_cases = [
    '{"has_causal": true, "triples": []}',
    'Here is the result:\n{"has_causal": false, "triples": []}',
    '```json\n{"has_causal": true, "triples": []}\n```',
    'Sure!\n```\n{"has_causal": false, "triples": []}\n```\nDone.',
    '<think>reasoning</think>\n{"has_causal": true, "triples": []}',
]
for i, tc in enumerate(test_cases, 1):
    try:
        result = parse_output(tc)
        print(f"OK 用例 {i}: {result}")
    except Exception as e:
        print(f"FAIL 用例 {i} 解析失败: {e}")


In [ ]:
# Cell L：单条 demo（manual 模式）
if MODE == "demo" and DEMO_INPUT == "manual":
    result = generate(
        text=DEMO_TEXT,
        sample_id=None,
        client=client,
        retriever=retriever if USE_RAG else None,
        use_rag=USE_RAG,
        top_k=RAG_TOP_K,
        rag_mode=RAG_MODE,
    )
    print("=== Demo 单条输出 ===")
    print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
# Cell M：数据集抽样 demo（sample 模式）
if MODE == "demo" and DEMO_INPUT == "sample":
    samples = load_dataset(DATASET, n=DEMO_SAMPLE_N)
    for s in samples:
        result = generate(
            text=s["text"],
            sample_id=s["id"],
            client=client,
            retriever=retriever if USE_RAG else None,
            use_rag=USE_RAG,
            top_k=RAG_TOP_K,
            rag_mode=RAG_MODE,
        )
        print(f"\n--- id={s['id']} ---")
        print(f"输入: {s['text']}")
        print(f"Gold: has_causal={s['has_causal']}, relations={len(s['relations'])}")
        print(f"Pred: {json.dumps(result, indent=2, ensure_ascii=False)}")


In [ ]:
# Cell N：批量生成（eval 模式预备，但本 SPEC 不做评估）
if MODE == "eval":
    samples = load_dataset(DATASET)
    predictions = []
    for s in samples[:20]:
        result = generate(
            text=s["text"],
            sample_id=s["id"],
            client=client,
            retriever=retriever if USE_RAG else None,
            use_rag=USE_RAG,
            top_k=RAG_TOP_K,
            rag_mode=RAG_MODE,
        )
        predictions.append({"pred": result, "gold": s})
    print(f"成功生成 {len(predictions)} 条预测，留待 SPEC_05 评估")
    for p in predictions[:3]:
        print(f"\nid={p['gold']['id']}")
        print(f"  Gold has_causal={p['gold']['has_causal']}, relations={p['gold']['relations']}")
        print(f"  Pred has_causal={p['pred']['has_causal']}, triples={p['pred']['triples']}")
